In [ ]:
library(STged)
library(DOTr)
library(ggplot2)
library(SeuratObject)
library(Seurat)
library(presto)
library(dplyr)
library(SeuratDisk)
library(Matrix)

In [ ]:
python_env <- "/net/data.isilon/ag-saez/yliu/SOFTWARE/.miniconda3/envs/multi/bin/python"
reticulate::use_python(python_env, required = TRUE)
reticulate::py_config()
anndata <- reticulate::import("anndata")
np <- reticulate::import("numpy")
sq <- reticulate::import("squidpy")

In [ ]:
st <-readRDS('data/spatial/processed_data/spatial.rds')

In [ ]:
donors = c('BCLL-8-T','BCLL-9-T','BCLL-10-T','BCLL-11-T','BCLL-12-T','BCLL-13-T')
prefix = c('c28w2r_7jne4i_','qvwc8t_2vsr67_','esvq52_nluss5_','exvyh1_66caqq_','p7hv1g_tjgmyj_','gcyl7c_cec61b_')

In [ ]:
donor <-'BCLL-8-T'
file <- paste0('data/spatial/ST_output/', 'BCLL-9-T', '.rds')
model.est <- readRDS(file)
#model.est <- readRDS('data/spatial/ST_output/donor_8.rds')
names(model.est)

In [ ]:
pred_expr_list <- model.est$F_list

In [ ]:
names(model.est$F_list)

In [ ]:
st_counts <- GetAssayData(st, assay = "Spatial", layer = "counts")
samples <- colnames(pred_expr_list[[1]])
genes <- rownames(pred_expr_list[[1]])
st_counts <- st_counts[genes, samples, drop = FALSE]
dim(st_counts)

In [ ]:
#file <- paste0('data/spatial/DOT_output/', donor, '.rds')
#dot <- readRDS(file)
beta <- model.est$beta
#weights_sub <- dot@weights[samples, , drop = FALSE]
#beta <- weights_sub / rowSums(weights_sub)

In [ ]:
names(pred_expr_list) <- colnames(beta)

In [ ]:
evaluate_target_ct_reliability <- function(
  st_exp,                # raw spatial expression: genes x spots
  prop_mat,              # cell type proportion: spots x celltypes
  pred_expr_list,        # list: each celltype has a predicted expr matrix (genes x spots)
  target_ct,             # target cell type name, e.g., “plasmablast”
  n_perm = 200,          # number of permutations
  genes_use = NULL,      # optional: only evaluate a subset of genes
  min_var_raw = 1e-6,    # minimum variance threshold for raw expression
  min_var_pred = 1e-6,   # minimum variance threshold for predicted expression (in target_ct)
  low_count_quantile = 0.01, # quantile threshold for low RNA count spots, e.g., 0.01 = remove lowest 1%
  verbose = TRUE
) {
  # ------------------ Basic checks ------------------
  ct_all <- colnames(prop_mat)
  if (is.null(ct_all)) {
    stop('prop_mat must have column names (cell type names)')
  }
  if (!all(ct_all %in% names(pred_expr_list))) {
    stop('pred_expr_list names must cover all column names of prop_mat (cell types)')
  }
  if (!(target_ct %in% ct_all)) {
    stop(paste0('target_ct = ', target_ct, 'is not in the column names of prop_mat'))
  }

  # ------------------ Record initial dimensions ------------------
  n_spots0 <- ncol(st_exp)
  n_genes0 <- nrow(st_exp)
  if (verbose) {
    message('Initial number of spots: ', n_spots0)
    message('Initial number of genes: ', n_genes0)
  }

  # ------------------ Spot filtering: low RNA count / low-quality spots ------------------
  # Use colSums of raw expression as a proxy
  total_counts <- colSums(st_exp, na.rm = TRUE)
  # If all are 0, then we cannot proceed
  if (all(total_counts == 0)) {
    stop('All spots have colSums(st_exp) equal to 0, cannot proceed.')
  }

  thresh <- quantile(total_counts, low_count_quantile, na.rm = TRUE)
  keep_spots <- which(total_counts >= thresh & total_counts > 0)

  if (length(keep_spots) < n_spots0) {
    if (verbose) {
      message('Filtering low RNA count / low-quality spots: removed ',
              n_spots0 - length(keep_spots),
              ' spots; kept ', length(keep_spots), ' spots.')
      message('Threshold used for rowSums(st_exp): ', signif(thresh, 4))
    }
  } else if (verbose) {
    message('No spots were filtered (all spots have rowSums >= quantile threshold).')
  }

  # Apply spot filtering to all matrices
  st_exp   <- st_exp[, keep_spots, drop = FALSE]
  prop_mat <- prop_mat[keep_spots, , drop = FALSE]
  for (ct in ct_all) {

    pred_expr_list[[ct]] <- pred_expr_list[[ct]][, keep_spots, drop = FALSE]
  }

  # ------------------ Gene alignment (rough) ------------------
  # First find the common genes among all cell types’ predicted expressions, then intersect with raw
  genes_pred_all <- Reduce(
    intersect,
    lapply(pred_expr_list[ct_all], function(m) rownames(m))
  )
  genes_common <- intersect(rownames(st_exp), genes_pred_all)
  if (!is.null(genes_use)) {
    genes_common <- intersect(genes_common, genes_use)
  }
  if (length(genes_common) == 0) {
    stop('No common genes found (in st_exp and all pred_expr_list).')
  }

  # Subset common genes
  st_exp   <- st_exp[genes_common, , drop = FALSE]
  for (ct in ct_all) {
    pred_expr_list[[ct]] <- pred_expr_list[[ct]][genes_common, , drop = FALSE]
  }

  if (verbose) {
    message('Number of genes after aligning with predicted expression: ', length(genes_common))
  }

  # ------------------ Gene filtering: raw all zero / low variance ------------------
  Y_raw <- as.matrix(st_exp)  # genes x spots

  raw_sum <- rowSums(Y_raw, na.rm = TRUE)
  raw_var <- apply(Y_raw, 1, var, na.rm = TRUE)
  keep_genes_raw <- which(raw_sum > 0 & raw_var > min_var_raw)

  if (length(keep_genes_raw) == 0) {
    stop('No genes remaining after filtering raw expression for all zero / low variance.')
  }

  if (verbose) {
    message('Filtering raw all zero or variance <', min_var_raw,
            ' genes: removed ', length(genes_common) - length(keep_genes_raw),
            ' genes; kept ', length(keep_genes_raw), ' genes.')
  }

  genes_common <- genes_common[keep_genes_raw]
  Y_raw        <- Y_raw[keep_genes_raw, , drop = FALSE]
  for (ct in ct_all) {
    pred_expr_list[[ct]] <- pred_expr_list[[ct]][keep_genes_raw, , drop = FALSE]
  }

  # ------------------ Gene filtering: Plasmablast predicted all zero / low variance ------------------
  pred_target <- as.matrix(pred_expr_list[[target_ct]])  # genes x spots
  pred_target_sum <- rowSums(pred_target, na.rm = TRUE)
  pred_target_var <- apply(pred_target, 1, var, na.rm = TRUE)

  keep_genes_plasma <- which(pred_target_sum > 0 & pred_target_var > min_var_pred)

  if (length(keep_genes_plasma) == 0) {
    stop('No genes remaining after filtering predicted all zero / low variance in target cell type (', target_ct, ').')
  }

  if (verbose) {
    message('Filtering predicted all zero or variance <', min_var_pred, ' genes in ', target_ct,
            ' genes: removed ', length(genes_common) - length(keep_genes_plasma),
            ' genes; finally kept ', length(keep_genes_plasma), ' genes.')
  }

  genes_common <- genes_common[keep_genes_plasma]
  Y_raw        <- Y_raw[keep_genes_plasma, , drop = FALSE]
  for (ct in ct_all) {
    pred_expr_list[[ct]] <- pred_expr_list[[ct]][keep_genes_plasma, , drop = FALSE]
  }

  # Final dimension report
  if (verbose) {
    message('Final number of spots used for evaluation: ', nrow(Y_raw))
    message('Final number of genes used for evaluation: ', ncol(Y_raw))
  }

  n_spots <- ncol(Y_raw)
  n_genes <- nrow(Y_raw)

  # ------------------ Calculate contribution matrix for each cell type: contr_ct ------------------
  contr_list <- list()
  for (ct in ct_all) {
    expr_ct <- as.matrix(pred_expr_list[[ct]])           # genes x spots
    prop_ct <- prop_mat[, ct]                            # length spots
    contr_ct <- expr_ct * rep(prop_ct, each = nrow(expr_ct))                        # broadcast by row
    contr_list[[ct]] <- contr_ct
  }

  # Contribution of target cell type and other cell types
  contr_all_sum   <- Reduce('+', contr_list[ct_all])     # genes x spots
  contr_target    <- contr_list[[target_ct]]             # genes x spots
  contr_other_sum <- contr_all_sum - contr_target        # genes x spots

  # ------------------ ΔR² + permutation ------------------
  res <- data.frame(
    gene     = genes_common,
    R2_0     = NA_real_,
    R2_1     = NA_real_,
    delta_R2 = NA_real_
    #p_value  = NA_real_
  )

  for (j in seq_len(n_genes)) {
    gname <- genes_common[j]
    y      <- Y_raw[j, ]                # raw expression
    y_hat0 <- contr_other_sum[j, ]      # prediction without target
    y_hat1 <- y_hat0 + contr_target[j, ]# prediction with target
    if (var(y) == 0) next  # just in case, check again

    sst  <- sum((y - mean(y))^2)
    sse0 <- sum((y - y_hat0)^2)
    sse1 <- sum((y - y_hat1)^2)

    R2_0 <- 1 - sse0 / sst
    R2_1 <- 1 - sse1 / sst
    delta_obs <- R2_1 - R2_0

    res$R2_0[j]     <- R2_0
    res$R2_1[j]     <- R2_1
    res$delta_R2[j] <- delta_obs

  #   # permutation: shuffle the spot order of target contribution
  #   if (n_perm > 0 && is.finite(delta_obs)) {
  #     delta_perm <- numeric(n_perm)
  #     ct_vec <- contr_target[j, ]

  #     for (b in seq_len(n_perm)) {
  #       idx <- sample.int(n_spots)
  #       ct_perm <- ct_vec[idx]
  #       y_hat1_perm <- y_hat0 + ct_perm

  #       sse1_perm <- sum((y - y_hat1_perm)^2)
  #       R2_1_perm <- 1 - sse1_perm / sst
  #       delta_perm[b] <- R2_1_perm - R2_0
  #     }

  #     p_val <- (sum(delta_perm >= delta_obs) + 1) / (n_perm + 1)
  #     res$p_value[j] <- p_val
  #   } else {
  #     res$p_value[j] <- NA_real_
  #   }
   }

  # res$reliability_score <- -log10(res$p_value)
  # res <- res[order(res$p_value), ]

  return(res)
}

In [ ]:
cell_types <- names(model.est$F_list)

In [ ]:
for (ct in cell_types) {
  res <- evaluate_target_ct_reliability(st_counts, beta, pred_expr_list, ct)
  write.csv(res,
          file = paste0("data/spatial/R_score/BCLL-9-T/", ct, ".csv"),
          row.names = FALSE)
}